In [ ]:
from google.oauth2.service_account import Credentials
import gspread
from gspread_dataframe import get_as_dataframe

from tabulate import tabulate
import pathlib
import shutil

import pygetwindow as gw
import pydirectinput as di
import pyautogui as ag
import pyperclip as pc
import time
import keyboard as kb

Important paths for the script.

In [ ]:
BASE_PATH            = pathlib.Path.cwd().resolve().parent.parent                  # ! <--- Will Change
DESKTOP_PATH         = pathlib.Path.home() / "Desktop"                             # * default export location
LOG_PATH             = DESKTOP_PATH / "Logs"                                       # ! <--- Will Change
ERROR_PATH           = BASE_PATH / "project" / "outputs" / "errors.txt"            #
SERVICE_ACCOUNT_PATH = BASE_PATH / "project" / "keys" / "service-account-key.json" #
RECORDING_PATH       = BASE_PATH / "project" / "outputs" / "recordings"            #

Hardcoded clickable locations in the CTS Player.

In [ ]:
UI_POSITIONS = {

    # Display
    "Display": (49,130),
    "Minimum": (184,333),
    "Center":  (310,333),
    "Maximum": (448,333),

    # Load Data
    "Load Data": (49,226),
    "Browse":    (490,100),
    "Path":      (890,72),
    "Accept":    (1060,672),
    "Select":    (350,154),

    # Advanced
    "Advanced":    (49,333),
    "Start frame": (280,170),
    "End frame":   (280,210),
    "Output FPS":  (280,250),
    "Export":      (180,105),

    # Preview
    "Loading":    (690,50),
    "Frame #":    (750,50),
    "Speed":      (1295,55),
    "2x":         (1290,160),
    "Play/Pause": (1353,55),

}

def reposition() -> None:
    window = gw.getWindowsWithTitle("CTS Playback v1.3.vi")[0]
    window.moveTo(0,0)

reposition()

Ensures that all of the correct settings are applied.

In [ ]:
UI_CHECKS = [

    # Display
    ((142,166), True),  # Upsample
    ((142,210), False), # Mirror
    ((405,255), False), # Dynamic
    ((137,640), True),  # High Contrast Mode
    ((340,640), True),  # Spatial Filter
    ((325,685), False), # Hotspot Detection

    # Preview
    ((1170,55),  False), # Loop
    ((1066,622), False), # Min
    ((1166,622), False), # Max
    ((1266,622), False), # Cursor

]

OFF_COLORS:list[tuple[int,int,int]] = [(45,45,45),(51,51,51)]

def check_UI():

    di.click(*UI_POSITIONS["Display"])

    for position, expected in UI_CHECKS:
        pixel:tuple[int,int,int] = ag.screenshot(region=(*position,1,1)).getpixel([0,0])
        current:bool = False if pixel in OFF_COLORS else True
        if current != expected:
            di.click(*position, interval=0.5)

    di.click(*UI_POSITIONS["Advanced"])

    di.doubleClick(*UI_POSITIONS["Output FPS"])
    di.write("4")

    di.click(*UI_POSITIONS["Speed"])
    di.doubleClick(*UI_POSITIONS["2x"], interval=0.5)

Entries represent individual falls/nonfalls and help translate the information from the google sheet to the corresponding AVI file.

In [ ]:
class Entry:

    def __init__(self,
        id:int,       #
        type:str,     #
        number:int,   #
        set:int,      #
        start:int,    #
        first:int,    #
        last:int,     #
        min:float,    #
        center:float, #
        max:float     #
    ):
        self.id = id
        self.type = type
        self.number = number
        self.set = set
        self.start = start
        self.first = first
        self.last = last
        self.min = min
        self.center = center
        self.max = max

    def __repr__(self) -> str:
        return f"{self.id:03}-{self.type}-{self.number:02}"

    def to_str(self) -> str:
        return f"{self} | {self.set} | {self.start}/{self.first}/{self.last} | {self.min}/{self.center}/{self.max}"

Moves the newly made avi file into correct location after recording.

In [ ]:
def relocate(entry:Entry) -> None:
    file = list(DESKTOP_PATH.glob("raw*.avi"))[0]
    name = f"{entry}.avi"
    path = RECORDING_PATH / str(entry.id)
    path.mkdir(exist_ok=True)
    shutil.move(str(file), path / name) # TODO: overwrite behavior

Record takes a given entry and performs the necesarry inputs to record it.

In [ ]:
def record(entry:Entry, set_temp:bool = False) -> None:

    if set_temp:
        di.click(*UI_POSITIONS["Display"])

        di.doubleClick(*UI_POSITIONS["Minimum"])
        pc.copy(str(entry.min))
        ag.hotkey("ctrl","v")

        di.doubleClick(*UI_POSITIONS["Center"])
        pc.copy(str(entry.center))
        ag.hotkey("ctrl","v")

        di.doubleClick(*UI_POSITIONS["Maximum"])
        pc.copy(str(entry.max))
        ag.hotkey("ctrl","v")

    di.click(*UI_POSITIONS["Advanced"])

    if ag.screenshot(region=(*UI_POSITIONS["Play/Pause"],1,1)).getpixel([0,0]) == (238,69,89):
        di.click(*UI_POSITIONS["Play/Pause"])

    di.doubleClick(*UI_POSITIONS["Frame #"])
    pc.copy(str(entry.start - 1))
    ag.hotkey("ctrl","v")

    di.doubleClick(*UI_POSITIONS["Start frame"])
    pc.copy(str(entry.start - 1))
    ag.hotkey("ctrl","v")

    di.doubleClick(*UI_POSITIONS["End frame"])
    pc.copy(str(entry.last - 1))
    ag.hotkey("ctrl","v")

    di.click(*UI_POSITIONS["Export"])

    while not gw.getWindowsWithTitle("Dialog-AviSaved.vi"):
        time.sleep(0.5)

    while gw.getWindowsWithTitle("Dialog-AviSaved.vi"):
        time.sleep(0.5)

    di.click(*UI_POSITIONS["Play/Pause"])

    relocate(entry)

Register takes a series of entries (recorded in the same set) and records them together.

In [ ]:
def register(series:list[Entry]):

    id:int = series[0].id
    type:int = series[0].type
    set:int = series[0].set

    di.click(*UI_POSITIONS["Load Data"])

    di.click(*UI_POSITIONS["Browse"])

    while not gw.getWindowsWithTitle("Choose or Enter Path of Folder"):
        time.sleep(0.5)

    window = gw.getWindowsWithTitle("Choose or Enter Path of Folder")[0]
    window.moveTo(0,0)

    path:str = f"{str(LOG_PATH)}\\{id}\\CTS-Data\\{type}\\set-{set}"
    pc.copy(path)

    di.click(*UI_POSITIONS["Path"])
    ag.hotkey("ctrl","v")

    di.press("enter")

    di.click(*UI_POSITIONS["Accept"])

    di.doubleClick(*UI_POSITIONS["Select"])

    while ag.screenshot(region=(*UI_POSITIONS["Loading"],1,1)).getpixel([0,0]) == (67,67,67):
        time.sleep(0.5)

    check_UI()

    current_temp:tuple[float,float,float] = (0.0,0.0,0.0)

    for entry in series:

        temp:tuple[float,float,float] = (entry.min,entry.center,entry.max)

        if temp != current_temp:
            record(entry, set_temp=True)
            current_temp = temp
        else:
            record(entry)

set_temp

In [ ]:
def set_temp(series:list[Entry]):

    id:int = series[0].id
    type:int = series[0].type
    set:int = series[0].set

    di.click(*UI_POSITIONS["Load Data"])

    di.click(*UI_POSITIONS["Browse"])

    while not gw.getWindowsWithTitle("Choose or Enter Path of Folder"):
        time.sleep(0.5)

    window = gw.getWindowsWithTitle("Choose or Enter Path of Folder")[0]
    window.moveTo(0,0)

    path:str = f"{str(LOG_PATH)}\\{id}\\CTS-Data\\{type}\\set-{set}"
    pc.copy(path)

    di.click(*UI_POSITIONS["Path"])
    ag.hotkey("ctrl","v")

    di.press("enter")

    di.click(*UI_POSITIONS["Accept"])

    di.doubleClick(*UI_POSITIONS["Select"])

    while ag.screenshot(region=(*UI_POSITIONS["Loading"],1,1)).getpixel([0,0]) == (67,67,67):
        time.sleep(0.5)

    check_UI()

    for entry in series:

        di.click(*UI_POSITIONS["Display"])

        di.doubleClick(*UI_POSITIONS["Minimum"])
        pc.copy(str(entry.min))
        ag.hotkey("ctrl","v")

        di.doubleClick(*UI_POSITIONS["Center"])
        pc.copy(str(entry.center))
        ag.hotkey("ctrl","v")

        di.doubleClick(*UI_POSITIONS["Maximum"])
        pc.copy(str(entry.max))
        ag.hotkey("ctrl","v")

        di.click(*UI_POSITIONS["Advanced"])

        if ag.screenshot(region=(*UI_POSITIONS["Play/Pause"],1,1)).getpixel([0,0]) == (238,69,89):
            di.click(*UI_POSITIONS["Play/Pause"])

        di.doubleClick(*UI_POSITIONS["Frame #"])
        pc.copy(str(entry.start - 1))
        ag.hotkey("ctrl","v")

        di.doubleClick(*UI_POSITIONS["Start frame"])
        pc.copy(str(entry.start - 1))
        ag.hotkey("ctrl","v")

        di.doubleClick(*UI_POSITIONS["End frame"])
        pc.copy(str(entry.last - 1))
        ag.hotkey("ctrl","v")

        di.click(*UI_POSITIONS["Display"])

        di.click(*UI_POSITIONS["Play/Pause"])

        # wait for "n" key press
        kb.wait('n')

This is how we access the google sheet and configure the data.

In [ ]:
SHEET_ID = "1-MfH3BwYLH2XqSgPD7GwxCQzV-k5PZv0_pGMzuYEiCI" # ! <--- Will Change
PAGE_NAME = "Fall"                                        # ! <--- Will Change

credentials = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_PATH,
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)

client = gspread.authorize(credentials)

sheet = client.open_by_key(SHEET_ID)

page = sheet.worksheet(PAGE_NAME)

data = get_as_dataframe(page)

data = data.dropna(subset=[
    "Start Frame",
    "First Frame",
    "Last Frame"
])

data["Recording Set"]    = data["Recording Set"].fillna(1)
data["Min Temp"]    = data["Min Temp"].fillna(0)
data["Center Temp"] = data["Center Temp"].fillna(22.5)
data["Max Temp"]    = data["Max Temp"].fillna(27.5)

# ! TEMP
# data = data[data["Volunteer ID"].astype(int) > 97]

data

Generates a list of series (which are themselves lists of entries) so that everything is in the proper form to start calling register.

In [ ]:
entries:list[list[Entry]] = []
current_id:int = int(data["Volunteer ID"].iloc[0])
current_set:int = int(data["Recording Set"].iloc[0])
series:list[Entry] = []

for _, row in data.iterrows():

    id:int = int(row["Volunteer ID"])
    set:int = int(row["Recording Set"])

    if id != current_id:
        current_id = id
        current_set = set
        entries.append(series)
        series = []

    if set != current_set:
        current_set = set
        entries.append(series)
        series = []

    type:str = PAGE_NAME
    number:int = int(row["Fall Number"])
    start:int = int(row["Start Frame"])
    first:int = int(row["First Frame"])
    last:int = int(row["Last Frame"])

    if start > first or first > last:
        print("ERROR")
        continue

    minimum:float = row["Min Temp"]
    center:float  = row["Center Temp"]
    maximum:float = row["Max Temp"]

    if minimum > center or center > maximum:
        print("ERROR")
        continue

    if minimum != 0:
        print("ERROR")
        continue

    series.append(Entry(id, type, number, set, start, first, last, minimum, center, maximum))

if series:
    entries.append(series)

for set in entries:
    for entry in set:
        print(entry.to_str())
    print("= = = = =")

Watch the magic happen.

In [ ]:
# di.moveTo(*UI_POSITIONS["Browse"])

# kb.wait('n')

# di.moveTo(*UI_POSITIONS["Path"])

# kb.wait('n')

# di.moveTo(*UI_POSITIONS["Accept"])

# kb.wait('n')

# di.moveTo(*UI_POSITIONS["Select"])

In [ ]:
for series in entries:
    set_temp(series)

In [ ]:
# for series in entries:
#     register(series)